# AtlasGuard — N-ATLaS inference endpoint on a free GPU

This notebook serves the **official `NCAIR1/N-ATLaS` weights** (revision `e294476928ac…`) with the AtlasGuard
inference service and exposes it to your AtlasGuard API through a temporary Cloudflare tunnel.

**Before you run it**

1. On Hugging Face, open <https://huggingface.co/NCAIR1/N-ATLaS> and click **Agree and access repository** (approval is automatic).
   Create a *read* token at <https://huggingface.co/settings/tokens>.
2. **Kaggle (recommended):** *Settings → Accelerator → GPU T4 ×2*, *Internet → On* (requires a verified phone number),
   *Add-ons → Secrets →* add `HF_TOKEN`. Two T4s hold the float16 weights, as in the model card.
   **Colab:** *Runtime → Change runtime type → T4 GPU*, add `HF_TOKEN` under *Secrets* and grant notebook access.
   A single 15 GB T4 cannot hold float16 weights, so this notebook switches to 8-bit quantization there and records that in
   every response's provenance.
3. Run all cells. The last cell prints `NATLAS_BASE_URL` and `NATLAS_API_KEY` for your AtlasGuard API.

**Notes:** keep this tab open while evaluating — Kaggle/Colab sessions and quick tunnels are temporary. Use the Docker image on a
GPU VM for stable hosting. N-ATLaS is released under its own licence (1,000 active-user cap without a commercial licence);
review it before deploying.

*N-ATLaS is an initiative of the Federal Ministry of Communications, Innovation and Digital Economy, and powered by Awarri Technologies.*

## 1. Write the AtlasGuard inference service

In [ ]:
import os

FILES = {
 "natlas-inference/app/__init__.py": "\"\"\"AtlasGuard N-ATLaS inference service (model inference only; no business logic).\"\"\"\n\n__version__ = \"0.1.0\"\n",
 "natlas-inference/app/config.py": "\"\"\"Service configuration from environment variables.\"\"\"\n\nfrom __future__ import annotations\n\nimport os\nfrom dataclasses import dataclass, field\n\n\ndef _bool(name: str, default: bool) -> bool:\n    value = os.environ.get(name)\n    if value is None or value == \"\":\n        return default\n    return value.strip().lower() in {\"1\", \"true\", \"yes\", \"on\"}\n\n\ndef _int(name: str, default: int) -> int:\n    value = os.environ.get(name)\n    return int(value) if value else default\n\n\n@dataclass(frozen=True)\nclass Settings:\n    #: Hugging Face repository of the official weights.\n    model_id: str = \"NCAIR1/N-ATLaS\"\n    #: Commit to load. Pinning makes evaluations reproducible; empty means the default branch.\n    revision: str | None = None\n    #: Name reported in responses. Must match AtlasGuard's NATLAS_MODEL_ID.\n    served_model_name: str = \"NCAIR1/N-ATLaS\"\n    #: Hugging Face token of an account that accepted the N-ATLaS terms (gated repository).\n    hf_token: str | None = None\n    #: Bearer token AtlasGuard must send. Required unless allow_unauthenticated is true.\n    api_key: str | None = None\n    allow_unauthenticated: bool = False\n    #: none | 8bit | 4bit (bitsandbytes).\n    quantization: str = \"none\"\n    #: float16 (model card default) | bfloat16 | float32.\n    torch_dtype: str = \"float16\"\n    #: Fixed date for the chat template (\"09 Oct 2026\"); empty uses today, like the model card.\n    chat_template_date: str | None = None\n    max_concurrent_generations: int = 1\n    max_new_tokens_limit: int = 2048\n    max_input_chars: int = 16000\n    load_on_startup: bool = True\n    cors_origins: list[str] = field(default_factory=list)\n\n    @classmethod\n    def from_env(cls) -> \"Settings\":\n        return cls(\n            model_id=os.environ.get(\"NATLAS_MODEL_ID\", \"NCAIR1/N-ATLaS\"),\n            revision=os.environ.get(\"NATLAS_REVISION\") or None,\n            served_model_name=os.environ.get(\"SERVED_MODEL_NAME\", \"NCAIR1/N-ATLaS\"),\n            hf_token=os.environ.get(\"HF_TOKEN\") or os.environ.get(\"HUGGING_FACE_HUB_TOKEN\") or None,\n            api_key=os.environ.get(\"INFERENCE_API_KEY\") or None,\n            allow_unauthenticated=_bool(\"ALLOW_UNAUTHENTICATED\", False),\n            quantization=os.environ.get(\"QUANTIZATION\", \"none\").lower(),\n            torch_dtype=os.environ.get(\"TORCH_DTYPE\", \"float16\").lower(),\n            chat_template_date=os.environ.get(\"CHAT_TEMPLATE_DATE\") or None,\n            max_concurrent_generations=_int(\"MAX_CONCURRENT_GENERATIONS\", 1),\n            max_new_tokens_limit=_int(\"MAX_NEW_TOKENS_LIMIT\", 2048),\n            max_input_chars=_int(\"MAX_INPUT_CHARS\", 16000),\n            load_on_startup=_bool(\"LOAD_ON_STARTUP\", True),\n        )\n\n    def validate(self) -> None:\n        if not self.api_key and not self.allow_unauthenticated:\n            raise RuntimeError(\n                \"INFERENCE_API_KEY is not set. Set it (and give the same value to AtlasGuard as NATLAS_API_KEY), \"\n                \"or set ALLOW_UNAUTHENTICATED=true for strictly local testing.\"\n            )\n        if self.quantization not in {\"none\", \"8bit\", \"4bit\"}:\n            raise RuntimeError(\"QUANTIZATION must be one of: none, 8bit, 4bit\")\n        if self.torch_dtype not in {\"float16\", \"bfloat16\", \"float32\"}:\n            raise RuntimeError(\"TORCH_DTYPE must be one of: float16, bfloat16, float32\")\n",
 "natlas-inference/app/schemas.py": "\"\"\"Request/response models for the OpenAI-compatible subset AtlasGuard uses.\"\"\"\n\nfrom __future__ import annotations\n\nfrom typing import Literal\n\nfrom pydantic import BaseModel, Field\n\n\nclass ChatMessage(BaseModel):\n    role: Literal[\"system\", \"user\", \"assistant\"]\n    content: str = Field(max_length=16000)\n\n\nclass ChatCompletionRequest(BaseModel):\n    model: str | None = None\n    messages: list[ChatMessage] = Field(min_length=1, max_length=32)\n    temperature: float = Field(default=0.1, ge=0.0, le=2.0)\n    max_tokens: int = Field(default=512, ge=1, le=4096)\n    top_p: float | None = Field(default=None, gt=0.0, le=1.0)\n    repetition_penalty: float | None = Field(default=None, ge=1.0, le=2.0)\n    seed: int | None = Field(default=None, ge=0, le=2**31 - 1)\n    stream: bool = False\n\n\nclass GenerationResult(BaseModel):\n    text: str\n    prompt_tokens: int\n    completion_tokens: int\n    finish_reason: Literal[\"stop\", \"length\"]\n    inference_ms: int\n    chat_template_date: str\n",
 "natlas-inference/app/backend.py": "\"\"\"Model backends. `TransformersBackend` runs the official N-ATLaS weights; tests use a fake.\"\"\"\n\nfrom __future__ import annotations\n\nimport threading\nimport time\nfrom datetime import datetime, timezone\nfrom typing import Any, Protocol\n\nfrom .config import Settings\nfrom .schemas import ChatMessage, GenerationResult\n\n\nclass Backend(Protocol):\n    \"\"\"What the HTTP layer needs from a model runtime.\"\"\"\n\n    def load(self) -> None: ...\n\n    @property\n    def loaded(self) -> bool: ...\n\n    @property\n    def load_error(self) -> str | None: ...\n\n    def info(self) -> dict[str, Any]: ...\n\n    def generate(\n        self,\n        messages: list[ChatMessage],\n        *,\n        temperature: float,\n        max_tokens: int,\n        top_p: float | None,\n        repetition_penalty: float | None,\n        seed: int | None,\n    ) -> GenerationResult: ...\n\n\nclass TransformersBackend:\n    \"\"\"\n    Loads NCAIR1/N-ATLaS with Hugging Face Transformers following the official model card:\n    official tokenizer and chat template (with `date_string`), float16 weights by default,\n    `device_map=\"auto\"`, and prompt tokenisation with `add_special_tokens=False`.\n    \"\"\"\n\n    def __init__(self, settings: Settings) -> None:\n        self.settings = settings\n        self._model: Any = None\n        self._tokenizer: Any = None\n        self._loaded = False\n        self._load_error: str | None = None\n        self._resolved_revision: str | None = settings.revision\n        self._device: str | None = None\n        self._runtime: str | None = None\n        self._lock = threading.Lock()\n\n    @property\n    def loaded(self) -> bool:\n        return self._loaded\n\n    @property\n    def load_error(self) -> str | None:\n        return self._load_error\n\n    def load(self) -> None:\n        try:\n            import torch\n            import transformers\n            from transformers import AutoModelForCausalLM, AutoTokenizer\n\n            kwargs: dict[str, Any] = {\"revision\": self.settings.revision, \"token\": self.settings.hf_token}\n            tokenizer = AutoTokenizer.from_pretrained(self.settings.model_id, **kwargs)\n            dtype = getattr(torch, self.settings.torch_dtype)\n            model_kwargs: dict[str, Any] = {**kwargs, \"device_map\": \"auto\", \"torch_dtype\": dtype}\n            if self.settings.quantization in {\"8bit\", \"4bit\"}:\n                from transformers import BitsAndBytesConfig\n\n                model_kwargs[\"quantization_config\"] = (\n                    BitsAndBytesConfig(load_in_8bit=True)\n                    if self.settings.quantization == \"8bit\"\n                    else BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=dtype)\n                )\n            model = AutoModelForCausalLM.from_pretrained(self.settings.model_id, **model_kwargs)\n            model.eval()\n\n            self._resolved_revision = getattr(model.config, \"_commit_hash\", None) or self.settings.revision\n            if torch.cuda.is_available():\n                names = sorted({torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())})\n                self._device = f\"cuda x{torch.cuda.device_count()} ({', '.join(names)})\"\n            else:\n                self._device = \"cpu\"\n            self._runtime = f\"transformers {transformers.__version__}; torch {torch.__version__}\"\n            self._tokenizer = tokenizer\n            self._model = model\n            self._loaded = True\n        except Exception as exc:  # noqa: BLE001 - surfaced through /health\n            self._load_error = f\"{type(exc).__name__}: {exc}\"\n            raise\n\n    def info(self) -> dict[str, Any]:\n        context_length = None\n        if self._model is not None:\n            context_length = getattr(self._model.config, \"max_position_embeddings\", None)\n        return {\n            \"model_id\": self.settings.served_model_name,\n            \"source_repository\": self.settings.model_id,\n            \"revision\": self._resolved_revision,\n            \"dtype\": self.settings.torch_dtype,\n            \"quantization\": self.settings.quantization,\n            \"device\": self._device,\n            \"runtime\": self._runtime,\n            \"context_length\": context_length,\n        }\n\n    def _template_date(self) -> str:\n        return self.settings.chat_template_date or datetime.now(timezone.utc).strftime(\"%d %b %Y\")\n\n    def generate(\n        self,\n        messages: list[ChatMessage],\n        *,\n        temperature: float,\n        max_tokens: int,\n        top_p: float | None,\n        repetition_penalty: float | None,\n        seed: int | None,\n    ) -> GenerationResult:\n        if not self._loaded:\n            raise RuntimeError(\"model is not loaded\")\n        import torch\n\n        date_string = self._template_date()\n        prompt = self._tokenizer.apply_chat_template(\n            [m.model_dump() for m in messages],\n            add_generation_prompt=True,\n            tokenize=False,\n            date_string=date_string,\n        )\n        inputs = self._tokenizer(prompt, return_tensors=\"pt\", add_special_tokens=False).to(self._model.device)\n        prompt_tokens = int(inputs[\"input_ids\"].shape[1])\n\n        generate_kwargs: dict[str, Any] = {\n            \"max_new_tokens\": max_tokens,\n            \"use_cache\": True,\n            \"pad_token_id\": self._tokenizer.eos_token_id,\n        }\n        if repetition_penalty is not None:\n            generate_kwargs[\"repetition_penalty\"] = repetition_penalty\n        if temperature > 0:\n            generate_kwargs.update(do_sample=True, temperature=temperature)\n            if top_p is not None:\n                generate_kwargs[\"top_p\"] = top_p\n        else:\n            generate_kwargs[\"do_sample\"] = False\n\n        with self._lock:\n            if seed is not None:\n                torch.manual_seed(seed)\n            started = time.perf_counter()\n            with torch.inference_mode():\n                output = self._model.generate(**inputs, **generate_kwargs)\n            inference_ms = int((time.perf_counter() - started) * 1000)\n\n        new_tokens = output[0][prompt_tokens:]\n        completion_tokens = int(new_tokens.shape[0])\n        text = self._tokenizer.decode(new_tokens, skip_special_tokens=True).strip()\n        finish_reason = \"length\" if completion_tokens >= max_tokens else \"stop\"\n        return GenerationResult(\n            text=text,\n            prompt_tokens=prompt_tokens,\n            completion_tokens=completion_tokens,\n            finish_reason=finish_reason,\n            inference_ms=inference_ms,\n            chat_template_date=date_string,\n        )\n",
 "natlas-inference/app/main.py": "\"\"\"\nOpenAI-compatible HTTP API in front of the official N-ATLaS weights.\n\nEndpoints\n  GET  /health                public; reports whether the model is loaded\n  GET  /v1/models             authenticated; the served model\n  GET  /model-info            authenticated; revision, dtype, device, runtime (provenance)\n  POST /v1/chat/completions   authenticated; non-streaming chat completion\n\nThis service performs inference only. Authentication of end users, evaluation logic and\npersistence live in the AtlasGuard API, which calls this service with a bearer token.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport asyncio\nimport logging\nimport secrets\nimport threading\nimport time\nimport uuid\nfrom contextlib import asynccontextmanager\nfrom typing import Any\n\nfrom fastapi import Depends, FastAPI, Header, HTTPException, Request\nfrom fastapi.responses import JSONResponse\nfrom starlette.concurrency import run_in_threadpool\n\nfrom . import __version__\nfrom .backend import Backend, TransformersBackend\nfrom .config import Settings\nfrom .schemas import ChatCompletionRequest\n\nlogger = logging.getLogger(\"natlas_inference\")\n\n\ndef create_app(settings: Settings | None = None, backend: Backend | None = None) -> FastAPI:\n    settings = settings or Settings.from_env()\n    settings.validate()\n    backend = backend or TransformersBackend(settings)\n    semaphore = asyncio.Semaphore(settings.max_concurrent_generations)\n    state: dict[str, Any] = {\"started_at\": time.time()}\n\n    def load_in_background() -> None:\n        try:\n            logger.info(\"Loading %s (revision=%s)…\", settings.model_id, settings.revision or \"default branch\")\n            backend.load()\n            logger.info(\"Model loaded: %s\", backend.info())\n        except Exception:  # noqa: BLE001\n            logger.exception(\"Model failed to load\")\n\n    @asynccontextmanager\n    async def lifespan(_: FastAPI):\n        if settings.load_on_startup and not backend.loaded:\n            threading.Thread(target=load_in_background, name=\"model-loader\", daemon=True).start()\n        yield\n\n    app = FastAPI(title=\"AtlasGuard N-ATLaS inference service\", version=__version__, lifespan=lifespan, docs_url=None, redoc_url=None)\n\n    def require_api_key(authorization: str | None = Header(default=None)) -> None:\n        if settings.allow_unauthenticated:\n            return\n        expected = f\"Bearer {settings.api_key}\"\n        if authorization is None or not secrets.compare_digest(authorization.encode(), expected.encode()):\n            raise HTTPException(status_code=401, detail=\"invalid or missing bearer token\")\n\n    @app.middleware(\"http\")\n    async def request_id(request: Request, call_next):\n        rid = request.headers.get(\"x-request-id\") or str(uuid.uuid4())\n        started = time.perf_counter()\n        response = await call_next(request)\n        response.headers[\"x-request-id\"] = rid\n        logger.info(\n            \"%s %s %s %dms rid=%s\", request.method, request.url.path, response.status_code,\n            int((time.perf_counter() - started) * 1000), rid,\n        )\n        return response\n\n    @app.get(\"/health\")\n    def health() -> dict[str, Any]:\n        return {\n            \"status\": \"ready\" if backend.loaded else (\"error\" if backend.load_error else \"loading\"),\n            \"model_loaded\": backend.loaded,\n            \"model_id\": settings.served_model_name,\n            \"error\": backend.load_error,\n            \"uptime_seconds\": int(time.time() - state[\"started_at\"]),\n            \"service_version\": __version__,\n        }\n\n    @app.get(\"/v1/models\", dependencies=[Depends(require_api_key)])\n    def models() -> dict[str, Any]:\n        return {\n            \"object\": \"list\",\n            \"data\": [{\"id\": settings.served_model_name, \"object\": \"model\", \"owned_by\": \"NCAIR1 / Awarri\"}],\n        }\n\n    @app.get(\"/model-info\", dependencies=[Depends(require_api_key)])\n    def model_info() -> dict[str, Any]:\n        return {**backend.info(), \"loaded\": backend.loaded, \"service_version\": __version__}\n\n    @app.post(\"/v1/chat/completions\", dependencies=[Depends(require_api_key)])\n    async def chat_completions(body: ChatCompletionRequest, request: Request) -> JSONResponse:\n        if body.stream:\n            raise HTTPException(status_code=400, detail=\"streaming is not supported\")\n        if body.model and body.model.lower() != settings.served_model_name.lower():\n            raise HTTPException(status_code=404, detail=f\"model '{body.model}' is not served here\")\n        if not backend.loaded:\n            raise HTTPException(status_code=503, detail=backend.load_error or \"model is still loading\")\n        if body.max_tokens > settings.max_new_tokens_limit:\n            raise HTTPException(status_code=422, detail=f\"max_tokens must be <= {settings.max_new_tokens_limit}\")\n        if sum(len(m.content) for m in body.messages) > settings.max_input_chars:\n            raise HTTPException(status_code=413, detail=\"input is too long\")\n\n        async with semaphore:\n            try:\n                result = await run_in_threadpool(\n                    backend.generate,\n                    body.messages,\n                    temperature=body.temperature,\n                    max_tokens=body.max_tokens,\n                    top_p=body.top_p,\n                    repetition_penalty=body.repetition_penalty,\n                    seed=body.seed,\n                )\n            except Exception as exc:  # noqa: BLE001\n                logger.exception(\"generation failed\")\n                raise HTTPException(status_code=500, detail=\"generation failed\") from exc\n\n        info = backend.info()\n        return JSONResponse(\n            {\n                \"id\": f\"chatcmpl-{uuid.uuid4().hex}\",\n                \"object\": \"chat.completion\",\n                \"created\": int(time.time()),\n                \"model\": settings.served_model_name,\n                \"choices\": [\n                    {\n                        \"index\": 0,\n                        \"message\": {\"role\": \"assistant\", \"content\": result.text},\n                        \"finish_reason\": result.finish_reason,\n                    }\n                ],\n                \"usage\": {\n                    \"prompt_tokens\": result.prompt_tokens,\n                    \"completion_tokens\": result.completion_tokens,\n                    \"total_tokens\": result.prompt_tokens + result.completion_tokens,\n                },\n                \"atlasguard\": {\n                    \"inference_ms\": result.inference_ms,\n                    \"revision\": info.get(\"revision\"),\n                    \"dtype\": info.get(\"dtype\"),\n                    \"quantization\": info.get(\"quantization\"),\n                    \"device\": info.get(\"device\"),\n                    \"runtime\": info.get(\"runtime\"),\n                    \"chat_template_date\": result.chat_template_date,\n                    \"request_id\": request.headers.get(\"x-request-id\"),\n                },\n            }\n        )\n\n    return app\n\n\ndef app_factory() -> FastAPI:\n    \"\"\"Entry point for `uvicorn app.main:app_factory --factory`.\"\"\"\n    logging.basicConfig(level=logging.INFO, format=\"%(asctime)s %(levelname)s %(name)s %(message)s\")\n    return create_app()\n"
}

for path, content in FILES.items():
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as fh:
        fh.write(content)
print("Wrote", len(FILES), "service files to", os.path.abspath("natlas-inference"))

## 2. Install dependencies

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "fastapi>=0.115", "uvicorn[standard]>=0.30",
                "transformers>=4.45,<5", "accelerate>=0.34", "huggingface_hub>=0.25"], check=True)
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name} — {p.total_memory / 1024**3:.1f} GB")

## 3. Configure (Hugging Face token, API key, precision)

In [ ]:
import os, secrets, torch

def read_secret(name):
    try:
        from kaggle_secrets import UserSecretsClient  # Kaggle
        return UserSecretsClient().get_secret(name)
    except Exception:
        pass
    try:
        from google.colab import userdata  # Colab
        return userdata.get(name)
    except Exception:
        pass
    return os.environ.get(name)

hf_token = read_secret("HF_TOKEN")
assert hf_token, "Add your Hugging Face read token as the secret HF_TOKEN (see the first cell)."

total_vram_gb = sum(torch.cuda.get_device_properties(i).total_memory for i in range(torch.cuda.device_count())) / 1024**3
quantization = "none" if total_vram_gb >= 20 else "8bit"
if quantization == "8bit":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes>=0.43"], check=True)

INFERENCE_API_KEY = secrets.token_urlsafe(32)
os.environ.update({
    "HF_TOKEN": hf_token,
    "INFERENCE_API_KEY": INFERENCE_API_KEY,
    "NATLAS_MODEL_ID": "NCAIR1/N-ATLaS",
    "NATLAS_REVISION": "e294476928aca9030e924ca27bb8e085e8581273",
    "SERVED_MODEL_NAME": "NCAIR1/N-ATLaS",
    "QUANTIZATION": quantization,
    "TORCH_DTYPE": "float16",
})
print(f"Total GPU memory: {total_vram_gb:.1f} GB -> quantization = {quantization}")

## 4. Start the service and load N-ATLaS (first download ≈ 16 GB, several minutes)

In [ ]:
import subprocess, time, json, urllib.request

log = open("natlas-inference/server.log", "w")
server = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "app.main:app_factory", "--factory", "--host", "127.0.0.1", "--port", "8000"],
    cwd="natlas-inference", stdout=log, stderr=subprocess.STDOUT,
)

def health():
    with urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=5) as r:
        return json.load(r)

started = time.time()
while True:
    time.sleep(10)
    try:
        h = health()
    except Exception:
        h = {"status": "starting"}
    print(f"[{int(time.time() - started):>4}s] {h.get('status')}" + (f" — {h.get('error')}" if h.get("error") else ""))
    if h.get("model_loaded") or h.get("status") == "error" or server.poll() is not None:
        break
print(open("natlas-inference/server.log").read()[-2000:])

## 5. Check provenance and run a test generation

In [ ]:
def call(path, payload=None):
    req = urllib.request.Request("http://127.0.0.1:8000" + path,
        data=json.dumps(payload).encode() if payload is not None else None,
        headers={"Authorization": f"Bearer {INFERENCE_API_KEY}", "Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=300) as r:
        return json.load(r)

print(json.dumps(call("/model-info"), indent=2))
reply = call("/v1/chat/completions", {
    "model": "NCAIR1/N-ATLaS",
    "messages": [{"role": "user", "content": "Kí ni olú-ìlú orílẹ̀-èdè Nàìjíríà?"}],
    "temperature": 0.1, "max_tokens": 128, "repetition_penalty": 1.12,
})
print(reply["choices"][0]["message"]["content"])
print(json.dumps(reply["usage"]), json.dumps(reply["atlasguard"]))

## 6. Expose the endpoint to AtlasGuard (Cloudflare quick tunnel)

In [ ]:
import re
subprocess.run("wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared",
               shell=True, check=True)
tunnel_log = open("cloudflared.log", "w")
tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8000"],
                          stdout=tunnel_log, stderr=subprocess.STDOUT)
url = None
for _ in range(60):
    time.sleep(2)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("cloudflared.log").read())
    if m:
        url = m.group(0)
        break
assert url, "Tunnel did not start; see cloudflared.log"
print("Add these to your AtlasGuard API environment (apps/api/.env or your host's settings):\n")
print(f"NATLAS_BASE_URL={url}")
print(f"NATLAS_API_KEY={INFERENCE_API_KEY}")
print("\nThen restart the API and open the dashboard: the model status should show N-ATLaS as available.")

## 7. Keep the session alive

Leave this notebook running while you evaluate. If the session restarts, run all cells again — you will get a **new**
tunnel URL and API key to update in AtlasGuard. Quick tunnels drop requests longer than ~100 s, so keep `maxTokens`
at 512 or below.